# 5.1 - Prompting Fundamentals

**Module 5 - Prompt Engineering** | Netsetos GenAI Engineering

This notebook turns prompting from guesswork into a measured engineering skill: you build one reusable `ask()` helper on Google's unified `google-genai` SDK, then run real API calls to see prompt anatomy, instruction clarity, roles, delimiters, structured output, reasoning styles, and reliability testing side by side. It ends with a production-shaped Product Review Analyzer that takes free-text reviews and emits clean, Pydantic-validated JSON.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - reproducibility marker

This first cell is a lone comment, not executable logic. It flags that the lesson's illustrative outputs were produced with seeded/low-temperature settings so results are repeatable - live API calls will still vary run to run.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A no-op comment cell. It documents intent (seeded, repeatable values throughout the lesson) but performs no computation and imports nothing.

**How the code works - step by step**
- A single `#` comment line - Python ignores it at runtime.
- Signals the pedagogical convention that follows: examples favor low/zero temperature for stable output.

**In one line:** a note to the reader, not code that runs.

**What you'll see:** (no output - it is a comment only)

## Setup - the `ask()` helper we reuse all lesson

Every example below funnels through one tiny wrapper around Gemini, built on the current unified `google-genai` SDK (the older `google.generativeai` package was deprecated on 2025-08-31). This is the environment prep plus the one function that makes the rest of the notebook a series of one-liners.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY` in the environment).

In [ ]:
# pip install "google-genai>=1.0.0"   (the unified SDK; replaces google.generativeai)
from google import genai
from google.genai import types
import os, time

# One client for the whole program. Reads GOOGLE_API_KEY from the environment.
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

def ask(prompt: str, temperature: float = 0.3) -> str:
    """Send one prompt to Gemini and return the text. Retries once on a transient error."""
    for attempt in range(2):
        try:
            resp = client.models.generate_content(
                model="gemini-3.5-flash",
                contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature),
            )
            return resp.text
        except Exception as e:
            if attempt == 1:
                return f"[API error after retry: {e}]"
            time.sleep(1.0)   # back off before retrying; real code retries ONLY transient 429/503/timeout, with exponential backoff

print(ask("Say hello in one word."))
# Output: Hello

**Explanation**

This is the plumbing, not a technique: it imports the SDK, builds one reusable client, and defines `ask()` - a thin send-a-prompt/get-text wrapper with a single retry so a network blip doesn't crash the script. Read it as the smallest honest production call pattern.

**How the code works - step by step**
- **`from google import genai` / `from google.genai import types`** - the unified package; `types` holds the config object for sampling and system instructions.
- **`genai.Client(api_key=...)`** - one client object built from `GOOGLE_API_KEY`, reused for every call (replaces the old `configure()` global plus per-call `GenerativeModel`).
- **`ask(prompt, temperature=0.3)`** - calls `client.models.generate_content(model="gemini-3.5-flash", contents=..., config=...)` and returns `resp.text`.
- **`try/except` with `range(2)`** - retries once on any exception, sleeping 1s before the second attempt; on a second failure it returns an `[API error...]` string instead of raising.
- **`print(ask("Say hello in one word."))`** - a smoke test that the key and client work.

**In one line:** build one client, wrap a single retry around one call, return text.

**What you'll see:** Prints `Hello` - a one-word confirmation that the API key and client are wired up correctly.

## 1 - Prompt anatomy: the four parts of every prompt

Every prompt - Gemini, Claude, or GPT - wears the same skeleton: instruction, context, input, output format. This cell contrasts an instruction-only prompt against one that pins all four parts, so you can feel how much the model was guessing before.

> **Needs a Gemini API key** (calls the live API once).

In [ ]:
# THE 4 PARTS OF EVERY PROMPT
#   1 INSTRUCTION   - what to do
#   2 CONTEXT       - who it is for, why, any background
#   3 INPUT         - the specific thing to act on
#   4 OUTPUT FORMAT - the exact shape of the answer

bad = "Tell me about Hyderabad"          # instruction only

good = """[INSTRUCTION] Write a travel-guide paragraph.
[CONTEXT]     For a first-time US visitor on a 3-day trip.
[INPUT]       City: Hyderabad, India.
[OUTPUT]      Top 3 attractions, best food, one insider tip. Under 150 words."""

print(ask(good))
# Output: Hyderabad packs history and flavor into three days. Start at the
# Output: Charminar and Golconda Fort... (a tight, on-brief 140-word guide)

**Explanation**

A conceptual demo, not a helper: two prompt strings and one API call. The comment block spells out the four-part checklist; `bad` supplies only an instruction while `good` labels all four parts explicitly, and only the good one is sent.

**How the code works - step by step**
- **Comment header** - names the four parts: INSTRUCTION (what), CONTEXT (who/why), INPUT (the specific thing), OUTPUT FORMAT (the shape).
- **`bad`** - `"Tell me about Hyderabad"`, instruction only; the other three parts are left for the model to guess.
- **`good`** - a triple-quoted string with `[INSTRUCTION]`/`[CONTEXT]`/`[INPUT]`/`[OUTPUT]` labels: travel paragraph, first-time US visitor, Hyderabad, top-3 + food + tip under 150 words.
- **`print(ask(good))`** - sends the fully-specified prompt.

**In one line:** label the four parts and nothing is left to guess.

**What you'll see:** A tight, on-brief ~140-word Hyderabad travel guide opening with the Charminar and Golconda Fort - concrete because all four parts were pinned.

## 2 - Instruction clarity: be specific, or get the average

The cheapest quality win in prompting: replace vague asks with explicit constraints. This cell runs the same underlying task - explain machine learning - as a vague prompt and a specific one, then prints both so the gap is visible.

> **Needs a Gemini API key** (two live calls).

In [ ]:
vague = "Tell me about machine learning."

specific = """Explain machine learning to a 15-year-old who has never heard of it.
Use exactly 3 sentences. Include one example they would relate to (Instagram
or YouTube). Do not use any technical jargon."""

print("=== VAGUE ==="); print(ask(vague))
print("=== SPECIFIC ==="); print(ask(specific))
# Output: === VAGUE ===   a 250-word textbook block, generic, no audience
# Output: === SPECIFIC === Machine learning is how apps learn what you like.
# Output: When Instagram shows you reels you enjoy, it learned from your taps...
# Output: (exactly 3 sentences, an Instagram example, zero jargon)

**Explanation**

A side-by-side comparison harness. `vague` gives one open-ended sentence; `specific` stacks four constraints (audience, exact length, a relatable example, a no-jargon rule). Both are sent and printed under labels.

**How the code works - step by step**
- **`vague`** - `"Tell me about machine learning."`, no audience, length, or format.
- **`specific`** - four constraints in one prompt: a 15-year-old audience, exactly 3 sentences, an Instagram/YouTube example, and no technical jargon.
- **`print("=== VAGUE ==="); print(ask(vague))`** then the same for `specific` - runs each and labels the output.

**In one line:** each added constraint narrows the answer toward the one you actually want.

**What you'll see:** The vague call returns a generic ~250-word textbook block; the specific call returns exactly 3 jargon-free sentences built around an Instagram example.

## 3 - Roles and system prompts: tell the model who it is

A one-sentence role shifts the model's vocabulary, depth, and default assumptions - and in production you set it once in the system prompt rather than repeating it every turn. This cell casts the model as a senior backend engineer and asks a generic question to see the persona take over.

> **Needs a Gemini API key** (one live call).

In [ ]:
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# The role is set ONCE in the config and applies to every call that uses it.
config = types.GenerateContentConfig(
    system_instruction=(
        "You are a senior backend engineer with 12 years building production "
        "Python APIs that serve millions of requests a day. Answer with the "
        "patterns you would actually ship: be concrete, name the failure modes, "
        "and keep it under 150 words."
    ),
    temperature=0.3,
)

resp = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="How should I handle errors in my Python API?",
    config=config,
)
print(resp.text)
# Output: Wrap handlers in typed exceptions, not bare except. Return structured
# Output: error envelopes with a code + request_id. Add timeouts and bounded
# Output: retries with backoff. Log with context; emit metrics; alert on 5xx rate...

**Explanation**

Shows where the role actually lives: inside `GenerateContentConfig(system_instruction=...)`, built once and reusable. The role conditions the answer without adding new facts - it selects which slice of existing knowledge the model writes from. Note this cell re-imports and rebuilds its own client so it stands alone.

**How the code works - step by step**
- **Re-imports + `client = genai.Client(...)`** - makes the cell self-contained.
- **`config = types.GenerateContentConfig(system_instruction=..., temperature=0.3)`** - the persona (senior backend engineer, concrete, name failure modes, under 150 words) is set once here, not in the user message.
- **`client.models.generate_content(contents="How should I handle errors...", config=config)`** - the task and data go in `contents`; the role stays in the config.
- **`print(resp.text)`** - shows the persona-shaped answer.

**In one line:** cast the model once in `system_instruction`, then reuse that config across calls.

**What you'll see:** Ships-it-tomorrow advice instead of a generic tutorial: typed exceptions, structured error envelopes with a code + request_id, bounded retries with backoff, metrics and 5xx alerting.

## 4 - Delimiters: separate data from instructions

Delimiters are industrial-strength quotation marks - a hard border between your instructions and the user's (untrusted) content, and the first line of injection defense. This cell shows the two provider-idiomatic fencing styles without spending an API call.

In [ ]:
# Claude is tuned toward XML tags; GPT toward markdown headers/fences. Gemini takes either.
claude_style = """Summarize the document in 3 sentences.
<document>
{user_text}
</document>"""

gpt_style = """### Instructions
Summarize the document in 3 sentences.
### Document
```
{user_text}
```"""

# Either way, the model now treats the fenced region as DATA, not commands.
print("Delimiters drawn: instructions and user data are now separated.")
# Output: Delimiters drawn: instructions and user data are now separated.

**Explanation**

A pure string-construction cell - no model call. It contrasts Claude-style XML tags with GPT-style markdown headers/fences (Gemini accepts either), both wrapping a `{user_text}` placeholder to mark it as data.

**How the code works - step by step**
- **`claude_style`** - wraps the user text in `<document>...</document>` XML tags, the format Claude is tuned toward.
- **`gpt_style`** - uses `### Instructions` / `### Document` markdown headers with a triple-backtick fence, the format GPT favors.
- **`print(...)`** - a status line; the point is the constructed strings, not a response.

**In one line:** fence the user's text so the model reads it as data, not commands.

**What you'll see:** Prints `Delimiters drawn: instructions and user data are now separated.` - no API call; the strings themselves are the lesson.

## 5 - Structured output: free text in, JSON out

The same fencing instinct that blocks injection also gets you machine-readable output - show the exact JSON shape and the model fills it in. This cell extracts a product review into typed JSON your code can actually parse.

> **Needs a Gemini API key** (one live call at temperature 0).

In [ ]:
import json

review = """Bought the Sony WH-1000XM5 for 29,999 rupees on Amazon. The noise
cancelling is incredible. Battery lasts forever. The touch controls are fiddly. 4/5."""

prompt = f"""Extract structured data from the review below.
Return ONLY valid JSON (no markdown, no commentary) with this exact shape:
{{"product": str, "price": number|null, "rating": number, "sentiment": "positive"|"negative"|"mixed", "cons": [str]}}

<review>
{review}
</review>"""

raw = ask(prompt, temperature=0.0).strip()   # temp 0 for stable, parseable output
if raw.startswith("```"):                        # defensive: strip a stray ```json fence before parsing
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
data = json.loads(raw)
print(data["product"], "|", data["rating"], "|", data["sentiment"])
# Output: Sony WH-1000XM5 | 4 | mixed

**Explanation**

An extraction cell: it hands the model a literal JSON skeleton (keys, types, enum values), fences the review, forces determinism, then defensively parses the result. The defensive fence-strip acknowledges that models often wrap JSON in markdown despite instructions.

**How the code works - step by step**
- **`review`** - a free-text Sony headphones review with price, pros, cons, and a 4/5 rating.
- **`prompt`** - an f-string that shows the exact shape `{"product":..., "price":..., "rating":..., "sentiment":..., "cons":[...]}` and says `Return ONLY valid JSON`, with the review inside `<review>` tags.
- **`ask(prompt, temperature=0.0)`** - temperature 0 for stable, parseable output.
- **`if raw.startswith("```")`** - strips a stray ` ```json ` fence before `json.loads`, so a rogue fence doesn't break parsing.
- **`json.loads(raw)` + `print(...)`** - parses and prints the key fields.

**In one line:** show the shape, force temperature 0, strip stray fences, parse.

**What you'll see:** Prints `Sony WH-1000XM5 | 4 | mixed` - the free-text review reduced to typed fields.

## 6 - Classic models: zero-shot chain-of-thought

On a classic instruction-following model, adding "Let's think step by step" is a real, measured accuracy win on multi-step problems (Kojima et al., 2022). This cell shows the move on a two-step discount-and-coupon math problem.

> **Needs a Gemini API key** (one live call).

In [ ]:
# CLASSIC MODEL: zero-shot chain-of-thought helps on multi-step math.
direct = "Shirt is 800 rupees, 25% off, then a 100-rupee coupon. Final price?"
cot    = direct + " Let's think step by step."

print(ask(cot, temperature=0.0))
# Output: 800 - 25% = 600. 600 - 100 coupon = 500. Final price: 500 rupees.

**Explanation**

A minimal demonstration of zero-shot chain-of-thought: take a direct question, append the trigger phrase, and let the model narrate its arithmetic. The point is the appended sentence, not the SDK call.

**How the code works - step by step**
- **`direct`** - the raw word problem: 800-rupee shirt, 25% off, then a 100-rupee coupon.
- **`cot`** - `direct + " Let's think step by step."`, the zero-shot CoT trigger.
- **`print(ask(cot, temperature=0.0))`** - runs the CoT version deterministically.

**In one line:** on a classic model, the trigger phrase buys real accuracy on multi-step math.

**What you'll see:** Prints the worked steps and answer: `800 - 25% = 600. 600 - 100 coupon = 500. Final price: 500 rupees.`

## 7 - Sampling temperature: creativity vs determinism

Temperature controls how sharp or flat the next-token distribution is before a token is drawn - low is safe and repeatable, high is creative and riskier. This cell sweeps the same tagline prompt across three temperatures to make the dial tangible.

> **Needs a Gemini API key** (three live calls).

In [ ]:
prompt = "Write a tagline for a Hyderabad biryani restaurant."

for t in (0.0, 0.7, 1.4):
    print(t, "->", ask(prompt, temperature=t))
# Output: 0.0 -> Authentic Hyderabadi biryani, every single day.   (safe, repeatable)
# Output: 0.7 -> Dum-cooked tradition, served by the kilo.        (balanced, default)
# Output: 1.4 -> Saffron rebellion in a copper handi!            (creative, riskier)

**Explanation**

A small sweep loop, not a model change: one prompt run at three temperatures so you can watch output go from safe to wild. Remember temperature moves randomness, not correctness - it can't fix a wrong answer.

**How the code works - step by step**
- **`prompt`** - `"Write a tagline for a Hyderabad biryani restaurant."`
- **`for t in (0.0, 0.7, 1.4)`** - iterates three temperatures spanning deterministic, balanced, and high-variance.
- **`print(t, "->", ask(prompt, temperature=t))`** - runs and labels each.

**In one line:** same prompt, three temperatures, three levels of risk.

**What you'll see:** Three taglines increasing in creativity: `0.0 ->` a safe/repeatable line, `0.7 ->` a balanced default, `1.4 ->` a florid, riskier one (e.g. "Saffron rebellion in a copper handi!").

## 8 - Reliability: run it N times and measure the pass rate

A prompt is code, so test it like code - one lucky run is not evidence. This cell builds a tiny harness that runs a prompt many times and reports how often each check passes, surfacing the drift a single run hides.

> **Needs a Gemini API key** (makes `runs` live calls - 10 by default).

In [ ]:
import json

def measure(prompt, checks, runs=10):
    """Run a prompt many times; report how often each check passes."""
    passes = {name: 0 for name in checks}
    for _ in range(runs):
        out = ask(prompt, temperature=0.7)   # >0 so runs actually vary - that variance is what we measure
        for name, check in checks.items():
            try:
                if check(out): passes[name] += 1
            except Exception:
                pass   # a thrown check counts as a fail
    return {name: f"{c}/{runs}" for name, c in passes.items()}

prompt = 'Return ONLY JSON: {"sentiment":"positive"|"negative","rating":1-5}. ' \
         'Review: "Great battery, dim screen. 3 stars."'

checks = {
    "valid_json": lambda r: json.loads(r) is not None,
    "has_rating": lambda r: "rating" in json.loads(r),
    "rating_is_3": lambda r: json.loads(r).get("rating") == 3,
}
print(measure(prompt, checks))
# Output: {'valid_json': '10/10', 'has_rating': '10/10', 'rating_is_3': '8/10'}
# The 8/10 is the real signal: sometimes the model reads "3 stars" as mixed=2.5.

**Explanation**

A measurement harness, not a model call in itself: `measure()` loops a prompt N times at nonzero temperature (so runs actually vary) and tallies a dict of named boolean checks into pass counts. The nonzero temperature is deliberate - the variance is exactly what's being measured.

**How the code works - step by step**
- **`measure(prompt, checks, runs=10)`** - initializes a per-check counter, then loops `runs` times calling `ask(..., temperature=0.7)`.
- **Inner check loop** - runs each `name: check` lambda on the output; a passing check increments its counter, and a thrown exception is swallowed as a fail.
- **`prompt`** - asks for JSON sentiment + rating on a mixed "3 stars" review.
- **`checks`** - three lambdas: `valid_json` (parses), `has_rating` (key present), `rating_is_3` (value correct).
- **`print(measure(prompt, checks))`** - reports each check as `passes/runs`.

**In one line:** run N times, count passes per check, read the pass rate.

**What you'll see:** A dict like `{'valid_json': '10/10', 'has_rating': '10/10', 'rating_is_3': '8/10'}` - the 8/10 is the real signal, showing the model occasionally misreads "3 stars."

## 9 - Capstone: the Product Review Analyzer

Every technique in one function: a role in the system prompt, clear rules, delimited input, structured output, and a validate-plus-retry loop so malformed JSON never reaches your database. Free-text reviews go in; clean, typed, Pydantic-validated data comes out.

> **Needs a Gemini API key** (one or two live calls per review).

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import os, json

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# 1. The exact output contract, as a Pydantic model.
class ReviewAnalysis(BaseModel):
    product: str
    rating: int = Field(ge=1, le=5)
    sentiment: str                # positive | negative | mixed
    pros: list[str]
    cons: list[str]
    recommend: bool

# 2. The role lives in the system instruction; set once.
config = types.GenerateContentConfig(
    system_instruction=(
        "You are a precise e-commerce review analyst. You output ONLY valid JSON "
        "matching the requested schema - no markdown fences, no commentary."
    ),
    temperature=0.0,
)

def analyze(review: str) -> ReviewAnalysis:
    prompt = f"""Extract data from the review. Return ONLY JSON matching:
{json.dumps(ReviewAnalysis.model_json_schema())}
Rules: rating is 1-5; sentiment is positive|negative|mixed; recommend is true if rating >= 3.
<review>{review}</review>"""
    for attempt in range(2):                       # 3. validate, and retry once on bad JSON
        try:
            raw = client.models.generate_content(
                model="gemini-3.5-flash", contents=prompt, config=config).text.strip()
            if raw.startswith("```"):
                raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
            return ReviewAnalysis.model_validate_json(raw)
        except Exception:
            if attempt == 1: raise

r = analyze("OnePlus Buds 3, 3299 rupees. Great bass and ANC, but they fall out when I jog. 4 stars.")
print(r.product, "|", r.rating, "|", r.sentiment, "|", r.recommend)
# Output: OnePlus Buds 3 | 4 | mixed | True

**Explanation**

The production-shaped payoff, assembling the whole lesson. A Pydantic model is the output contract, the role is set once in the config, and `analyze()` builds a schema-driven prompt, strips stray fences, validates against the model, and retries once on bad JSON before raising.

**How the code works - step by step**
- **`class ReviewAnalysis(BaseModel)`** - the exact contract: product, `rating` constrained to 1-5 via `Field(ge=1, le=5)`, sentiment, pros/cons lists, and a `recommend` bool.
- **`config = GenerateContentConfig(system_instruction=..., temperature=0.0)`** - casts the model as a precise JSON-only review analyst, set once.
- **`analyze(review)`** - builds a prompt that embeds `ReviewAnalysis.model_json_schema()`, states the rules (rating 1-5, sentiment enum, recommend if rating >= 3), and fences the review in `<review>` tags.
- **Retry loop (`for attempt in range(2)`)** - calls the API, strips a stray ` ``` ` fence, and returns `ReviewAnalysis.model_validate_json(raw)`; on the second failed attempt it re-raises.
- **`r = analyze(...)` + `print(...)`** - runs it on a OnePlus Buds review and prints validated fields.

**In one line:** role + rules + delimiters + schema + validate-and-retry = free text in, typed object out.

**What you'll see:** Prints `OnePlus Buds 3 | 4 | mixed | True` - a validated `ReviewAnalysis` object, guaranteed to match the schema or the call would have raised.

Across nine runnable cells you moved from a single `ask()` helper to a full free-text-in, typed-data-out pipeline. The throughline: a prompt is code - anatomy gives it a skeleton, clarity removes the model's room to wander, a role selects which knowledge it writes from, delimiters and JSON shape make output machine-readable, the classic-vs-reasoning split tells you whether to narrate the process or just name the goal, and a reliability harness turns "it worked once" into a pass rate you can gate a deploy on. Next up: Lesson 5.2 (few-shot and in-context learning) adds worked examples to the prompt, and Module 14 scales the N-times measurement idea into systematic eval-driven development.